# so-vits-svc-fork — Colab 학습 노트북 (GPU 없는 PC용)

**시작 전 필수**: 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택

**준비물** (Google Drive의 `MyDrive/svc_project/` 에 업로드):
- **A안**: 로컬에서 `preprocess.py`로 만든 `dataset_raw.zip` (내부 구조: `dataset_raw/가수명/*.wav`)
- **B안**: 전처리를 안 했다면 원본 곡들을 압축한 `songs.zip` → 5~6번 셀에서 Colab GPU로 분리+슬라이싱

**사용법**: 위에서 아래로 순서대로 실행. 세션이 끊기면 다시 연결 후 1~3번 셀 실행 → 학습 셀 재실행(자동으로 이어서 학습됨).

> 실존 가수 음성 학습은 본인 동의가 필요합니다. 연구·개인 학습 범위에서 사용하세요.


In [ ]:
# 1) GPU 확인
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다! 런타임 → 런타임 유형 변경 → T4 GPU'
print('GPU OK:', torch.cuda.get_device_name(0))


In [ ]:
# 2) Google Drive 마운트 + 작업 폴더
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/svc_project'   # 업로드한 zip과 학습 결과가 저장될 폴더
os.makedirs(WORK, exist_ok=True)
print('작업 폴더:', WORK)


In [ ]:
# 3) so-vits-svc-fork 설치 (2~3분)
# 이후 셀에서 import 오류가 나면: 런타임 재시작 후 1~3번 셀부터 다시 실행
!pip install -q -U so-vits-svc-fork
!svc --version


## 데이터 준비 — A안: 전처리된 dataset_raw.zip 사용 (권장)
B안으로 갈 경우 이 셀은 건너뛰세요.


In [ ]:
# 4) [A안] Drive의 dataset_raw.zip 압축 해제
%cd /content
!unzip -qo "{WORK}/dataset_raw.zip" -d /content/
!find dataset_raw -name "*.wav" | wc -l


## 데이터 준비 — B안: 원본 곡(songs.zip)만 있는 경우
Colab GPU로 보컬 분리(곡당 ~1분) 후 슬라이싱. **A안을 실행했다면 5~6번 셀은 건너뛰세요.**


In [ ]:
# 5) [B안] 보컬 분리 (BS-RoFormer)
!pip install -q "audio-separator[gpu]"
!unzip -qo "{WORK}/songs.zip" -d /content/songs

import glob
from audio_separator.separator import Separator
sep = Separator(output_dir='/content/separated')
sep.load_model(model_filename='model_bs_roformer_ep_317_sdr_12.9755.ckpt')
songs = sorted(glob.glob('/content/songs/**/*.mp3', recursive=True)
             + glob.glob('/content/songs/**/*.wav', recursive=True))
print(len(songs), '곡 발견')
for s in songs:
    print('분리 중:', s)
    sep.separate(s)


In [ ]:
# 6) [B안] 무음 기준 슬라이싱 → dataset_raw/가수명/
SINGER = 'singer'   # <- 가수 이름(영문)으로 변경

import os, glob
import numpy as np
import soundfile as sf

def slice_wav(src, dst_dir, base, threshold_db=-40.0, min_interval=0.3, min_keep=2.0, max_len=15.0):
    audio, sr = sf.read(src, always_2d=True)
    mono = audio.mean(axis=1)
    hop = int(sr * 0.010)
    n = len(mono) // hop
    if n == 0:
        return 0
    rms = np.sqrt(np.mean(mono[:n*hop].reshape(n, hop)**2, axis=1) + 1e-12)
    voiced = 20*np.log10(rms + 1e-12) > threshold_db
    segs, start = [], None
    for i, v in enumerate(voiced):
        if v and start is None:
            start = i
        elif not v and start is not None:
            segs.append([start, i]); start = None
    if start is not None:
        segs.append([start, n])
    gap = max(1, int(min_interval/0.010)); merged = []
    for s in segs:
        if merged and s[0]-merged[-1][1] <= gap:
            merged[-1][1] = s[1]
        else:
            merged.append(s)
    count, pad, chunk = 0, int(0.1/0.010), int(max_len*sr)
    for s0, s1 in merged:
        a, b = max(0, (s0-pad)*hop), min(len(mono), (s1+pad)*hop)
        pos = a
        while pos < b:
            end = min(pos+chunk, b)
            if (end-pos)/sr >= min_keep:
                sf.write(os.path.join(dst_dir, f'{base}_{count:04d}.wav'),
                         mono[pos:end], sr, subtype='PCM_16')
                count += 1
            pos = end
    return count

dst = f'/content/dataset_raw/{SINGER}'
os.makedirs(dst, exist_ok=True)
total = 0
for i, v in enumerate(sorted(glob.glob('/content/separated/*(Vocals)*.wav'))):
    total += slice_wav(v, dst, f'song{i:03d}')
print('총 클립:', total)


## 학습


In [ ]:
# 7) SVC 전처리 (리샘플 → 설정 생성 → HuBERT/f0 특징 추출)
%cd /content
!svc pre-resample
!svc pre-config
!svc pre-hubert -fm crepe


In [ ]:
# 8) 학습 설정 조정
import json
p = '/content/configs/44k/config.json'
cfg = json.load(open(p))
cfg['train']['epochs'] = 400        # 데이터 30분~1시간 기준. 부족하면 늘려서 재실행(이어서 학습됨)
cfg['train']['batch_size'] = 16     # T4 기준. 메모리 부족(OOM) 시 8로
json.dump(cfg, open(p, 'w'), indent=2)
print('epochs:', cfg['train']['epochs'], '/ batch_size:', cfg['train']['batch_size'])


In [ ]:
# 9) 학습 시작 — 체크포인트를 Drive에 저장 (사전학습 베이스 모델 자동 다운로드 후 fine-tuning)
# 세션이 끊기면 1~3번 셀 실행 후 이 셀만 다시 실행하면 마지막 체크포인트에서 이어짐
!mkdir -p "{WORK}/logs/44k"
!cp /content/configs/44k/config.json "{WORK}/logs/44k/config.json"
!svc train --model-path "{WORK}/logs/44k"


In [ ]:
# 10) (선택) 학습 곡선 확인 — 학습 중 별도 실행
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/svc_project/logs/44k


## 추론 (음색 변환)
변환할 커버 곡의 **보컬만 분리한 wav**를 Drive의 `svc_project/cover.wav` 로 업로드 후 실행.
(반주는 나중에 오디오 편집기에서 변환된 보컬과 합치면 됩니다)


In [ ]:
# 11) 변환 실행
import os, glob
ckpts = sorted(glob.glob(f'{WORK}/logs/44k/G_*.pth'), key=os.path.getmtime)
CKPT = ckpts[-1]
print('사용 체크포인트:', CKPT)
# -t: 키 조절(반음). 원곡 가수와 성별이 다르면 -12 또는 +12 시도
!svc infer "{WORK}/cover.wav" -m "{CKPT}" -c "{WORK}/logs/44k/config.json" -fm crepe -t 0 -o "{WORK}/cover_converted.wav"


In [ ]:
# 12) 결과 듣기 (파일은 Drive의 svc_project/cover_converted.wav 에 저장됨)
from IPython.display import Audio
Audio(f'{WORK}/cover_converted.wav')
